In [2]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from pathlib import Path
from datetime import timedelta
import math
import numpy as np
import polars as pl

from tqdm.auto import tqdm

PROJECT_PATH = Path(
    "/content/drive/MyDrive/multimodal-fashion-recsys"
)

PROCESSED_PATH = PROJECT_PATH / "data" / "processed"
SUBMISSIONS_PATH = PROJECT_PATH / "submissions"
EMBEDDINGS_PATH = PROJECT_PATH / "embeddings"

SUBMISSIONS_PATH.mkdir(
    parents=True,
    exist_ok=True
)

In [4]:
submission_candidates = list(
    SUBMISSIONS_PATH.glob("*.csv")
)

print("CSV files:")

for path in submission_candidates:
    print(path.name)

CSV files:
final_hybrid_submission.csv
submission.csv


In [5]:
template_submission = None
template_path = None

for path in submission_candidates:
    try:
        df = pl.read_csv(
            path,
            schema_overrides={
                "customer_id": pl.String,
                "prediction": pl.String
            }
        )

        if (
            df.height == 1371980
            and "customer_id" in df.columns
            and "prediction" in df.columns
        ):
            template_submission = df
            template_path = path
            break

    except Exception:
        pass

assert template_submission is not None

print("Template:", template_path.name)
print("Rows:", template_submission.height)
print(
    "Unique customers:",
    template_submission["customer_id"].n_unique()
)

Template: final_hybrid_submission.csv
Rows: 1371980
Unique customers: 1371980


In [6]:
transactions = pl.scan_parquet(
    PROCESSED_PATH / "transactions_mapped.parquet"
)

customer_mapping = pl.read_parquet(
    PROCESSED_PATH / "customer_mapping.parquet"
)

article_mapping = pl.read_parquet(
    PROCESSED_PATH / "article_mapping.parquet"
)

print("Customers:", customer_mapping.height)
print("Articles:", article_mapping.height)

Customers: 1371980
Articles: 105542


In [7]:
MAX_DATE = (
    transactions
    .select(
        pl.col("t_dat").max()
    )
    .collect()
    .item()
)

HISTORY_DAYS = 56
HALF_LIFE = 3

HISTORY_START = (
    MAX_DATE
    - timedelta(days=HISTORY_DAYS)
)

print("Last transaction:", MAX_DATE)
print("History start:", HISTORY_START)
print("Half-life:", HALF_LIFE)

Last transaction: 2020-09-22
History start: 2020-07-28
Half-life: 3


In [8]:
history_56d = (
    transactions
    .filter(
        pl.col("t_dat") >= HISTORY_START
    )
    .sort(
        ["customer_idx", "t_dat"],
        descending=[False, True]
    )
    .group_by(
        "customer_idx",
        maintain_order=True
    )
    .agg(
        pl.col("article_idx")
        .unique(
            maintain_order=True
        )
        .head(12)
        .alias("history")
    )
    .collect()
)

print(
    "Users with 56d history:",
    history_56d.height
)

history_56d.head()

Users with 56d history: 383797


customer_idx,history
u32,list[u32]
1,[16024]
3,[78504]
5,"[104813, 77916, … 101368]"
7,"[58296, 3092]"
14,"[14274, 104306, … 60793]"


In [9]:
decay_top100 = (
    transactions
    .with_columns(
        (
            -math.log(2)
            * (
                pl.lit(MAX_DATE)
                - pl.col("t_dat")
            )
            .dt.total_days()
            / HALF_LIFE
        )
        .exp()
        .alias("weight")
    )
    .group_by("article_idx")
    .agg(
        pl.col("weight")
        .sum()
        .alias("score")
    )
    .sort(
        "score",
        descending=True
    )
    .head(100)
    .collect()["article_idx"]
    .to_list()
)

print("DecayPop items:", len(decay_top100))
print(decay_top100[:12])

DecayPop items: 100
[104554, 104555, 104073, 67523, 3092, 104528, 103797, 95500, 103109, 56695, 104046, 71108]


In [10]:
submission_users = (
    template_submission
    .select("customer_id")
    .with_row_index("_row")
    .join(
        customer_mapping,
        on="customer_id",
        how="left"
    )
    .join(
        history_56d,
        on="customer_idx",
        how="left"
    )
    .sort("_row")
)

assert submission_users.height == 1371980
assert (
    submission_users["customer_idx"]
    .null_count()
    == 0
)

print("Users:", submission_users.height)
print(
    "Users with history:",
    submission_users["history"]
    .is_not_null()
    .sum()
)

Users: 1371980
Users with history: 383797


In [11]:
article_id_strings = [
    ""
] * (
    article_mapping["article_idx"].max()
    + 1
)

for article_idx, article_id in (
    article_mapping
    .select(
        "article_idx",
        "article_id"
    )
    .iter_rows()
):
    article_id_strings[
        article_idx
    ] = str(article_id).zfill(10)

print(article_id_strings[1:6])

['0108775015', '0108775044', '0108775051', '0110065001', '0110065002']


In [12]:
history_lists = (
    submission_users["history"]
    .to_list()
)

history_decay_predictions = []

for history in tqdm(
    history_lists,
    desc="History + DecayPop"
):
    recommendations = []
    used = set()

    if history is not None:
        for item in history:
            item = int(item)

            if item not in used:
                recommendations.append(item)
                used.add(item)

            if len(recommendations) == 12:
                break

    if len(recommendations) < 12:
        for item in decay_top100:
            item = int(item)

            if item not in used:
                recommendations.append(item)
                used.add(item)

            if len(recommendations) == 12:
                break

    prediction = " ".join(
        article_id_strings[item]
        for item in recommendations
    )

    history_decay_predictions.append(
        prediction
    )

History + DecayPop:   0%|          | 0/1371980 [00:00<?, ?it/s]

In [13]:
history_decay_submission = (
    pl.DataFrame({
        "customer_id":
            template_submission[
                "customer_id"
            ],

        "prediction":
            history_decay_predictions
    })
)

history_decay_submission.head()

customer_id,prediction
str,str
"""00000dbacae5abe5e23885899a1fa4…","""0568601043 0924243001 09242430…"
"""0000423b00ade91418cceaf3b26c6a…","""0924243001 0924243002 09185220…"
"""000058a12d5b43e67d225668fa1f8d…","""0794321007 0924243001 09242430…"
"""00005ca1c9ed5f5146b52ac8639a40…","""0924243001 0924243002 09185220…"
"""00006413d8573cd20ed7128e53b7b1…","""0927530004 0791587015 07306830…"


In [14]:
prediction_lengths = (
    history_decay_submission[
        "prediction"
    ]
    .str.split(" ")
    .list.len()
)

assert (
    history_decay_submission.height
    == 1371980
)

assert (
    history_decay_submission[
        "customer_id"
    ]
    .n_unique()
    == 1371980
)

assert (
    history_decay_submission[
        "customer_id"
    ]
    == template_submission[
        "customer_id"
    ]
).all()

assert prediction_lengths.min() == 12
assert prediction_lengths.max() == 12

print(
    "History + DecayPop submission is ready"
)

History + DecayPop submission is ready


In [15]:
HISTORY_DECAY_PATH = (
    SUBMISSIONS_PATH
    / "submission_history_decay.csv"
)

history_decay_submission.write_csv(
    HISTORY_DECAY_PATH
)

print(
    "Saved:",
    HISTORY_DECAY_PATH
)

print(
    f"Size: "
    f"{HISTORY_DECAY_PATH.stat().st_size / 1024**2:.2f} MB"
)

Saved: /content/drive/MyDrive/multimodal-fashion-recsys/submissions/submission_history_decay.csv
Size: 257.76 MB


In [16]:
sasrec_files = sorted(
    set(
        list(
            EMBEDDINGS_PATH.glob(
                "*sasrec*.npz"
            )
        )
        + list(
            PROJECT_PATH.rglob(
                "*sasrec*top100*.npz"
            )
        )
    )
)

print("SASRec files:")

for path in sasrec_files:
    print(path)

SASRec files:
/content/drive/MyDrive/multimodal-fashion-recsys/embeddings/sasrec_validation_top100.npz


In [17]:
for path in sasrec_files:
    try:
        data = np.load(path)

        print()
        print(path.name)
        print("Keys:", data.files)

        for key in data.files:
            print(
                key,
                data[key].shape
            )

    except Exception as error:
        print(
            path.name,
            error
        )


sasrec_validation_top100.npz
Keys: ['customer_idx', 'recommendations']
customer_idx (72019,)
recommendations (72019, 100)


In [18]:
RECENT_DAYS = 14

RECENT_START = (
    MAX_DATE
    - timedelta(days=RECENT_DAYS)
)

recent_top100 = (
    transactions
    .filter(
        pl.col("t_dat") >= RECENT_START
    )
    .group_by("article_idx")
    .agg(
        pl.len().alias("count")
    )
    .sort(
        "count",
        descending=True
    )
    .head(100)
    .collect()["article_idx"]
    .to_list()
)

print("Recent start:", RECENT_START)
print("Recent items:", len(recent_top100))
print(recent_top100[:12])

Recent start: 2020-09-08
Recent items: 100
[103109, 104554, 104073, 95218, 67523, 3092, 71108, 104046, 104528, 104555, 103797, 91738]


In [19]:
history_recent_predictions = []

for history in tqdm(
    history_lists,
    desc="History + RecentPop"
):
    recommendations = []
    used = set()

    if history is not None:
        for item in history:
            item = int(item)

            if item not in used:
                recommendations.append(item)
                used.add(item)

            if len(recommendations) == 12:
                break

    if len(recommendations) < 12:
        for item in recent_top100:
            item = int(item)

            if item not in used:
                recommendations.append(item)
                used.add(item)

            if len(recommendations) == 12:
                break

    prediction = " ".join(
        article_id_strings[item]
        for item in recommendations
    )

    history_recent_predictions.append(
        prediction
    )

History + RecentPop:   0%|          | 0/1371980 [00:00<?, ?it/s]

In [20]:
history_recent_submission = pl.DataFrame({
    "customer_id":
        template_submission["customer_id"],

    "prediction":
        history_recent_predictions
})

history_recent_submission.head()

customer_id,prediction
str,str
"""00000dbacae5abe5e23885899a1fa4…","""0568601043 0909370001 09242430…"
"""0000423b00ade91418cceaf3b26c6a…","""0909370001 0924243001 09185220…"
"""000058a12d5b43e67d225668fa1f8d…","""0794321007 0909370001 09242430…"
"""00005ca1c9ed5f5146b52ac8639a40…","""0909370001 0924243001 09185220…"
"""00006413d8573cd20ed7128e53b7b1…","""0927530004 0791587015 07306830…"


In [21]:
prediction_lengths = (
    history_recent_submission["prediction"]
    .str.split(" ")
    .list.len()
)

assert history_recent_submission.height == 1371980

assert (
    history_recent_submission["customer_id"]
    .n_unique()
    == 1371980
)

assert (
    history_recent_submission["customer_id"]
    == template_submission["customer_id"]
).all()

assert prediction_lengths.min() == 12
assert prediction_lengths.max() == 12

print("History + RecentPop submission is ready")

History + RecentPop submission is ready


In [22]:
HISTORY_RECENT_PATH = (
    SUBMISSIONS_PATH
    / "submission_history_recent14.csv"
)

history_recent_submission.write_csv(
    HISTORY_RECENT_PATH
)

print("Saved:", HISTORY_RECENT_PATH)

print(
    f"Size: "
    f"{HISTORY_RECENT_PATH.stat().st_size / 1024**2:.2f} MB"
)

Saved: /content/drive/MyDrive/multimodal-fashion-recsys/submissions/submission_history_recent14.csv
Size: 257.76 MB
